# Editing a model and its Rashomon view

A small sum is easy to change by hand: round a threshold, add a business rule, force a
monotone effect. After an edit, `refit_leaves` re-estimates the leaf values for the new
structure, and every output (rules, contributions, JSON) follows.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

from credit_data import make_credit_data

X, y = make_credit_data(n=20_000, seed=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
print(f"{len(X_train):,} training rows, {len(X_test):,} test rows, "
      f"{y.mean():.1%} defaults, {X['income'].isna().mean():.0%} incomes missing")

14,000 training rows, 6,000 test rows, 10.3% defaults, 5% incomes missing


In [2]:
from bettertrees import InterleavedTreeClassifier


def score(model):
    p = model.predict_proba(X_test)[:, 1]
    return f"AUC {roc_auc_score(y_test, p):.4f}, log-loss {log_loss(y_test, p):.4f}"


itm = InterleavedTreeClassifier(max_splits=16).fit(X_train, y_train)
print("fitted:", score(itm))
print(itm.explain())

fitted: AUC 0.7029, log-loss 0.3072
logit P(y = 1) = base -2.1659 + sum of 8 trees (16 cuts)
tree 1:
  -0.3639  if late_payments <= 0.5
  +0.3077  if late_payments > 0.5 and debt_ratio <= 0.4325
  +0.9798  if late_payments > 0.5 and debt_ratio > 0.4325
tree 2:
  +0.4033  if (income <= 3.635e+04 or missing) and credit_lines <= 4.5
  +0.0971  if (income <= 3.635e+04 or missing) and credit_lines > 4.5
  -0.0432  if 3.635e+04 < income <= 5.085e+04
  -0.5047  if income > 5.085e+04
tree 3:
  -0.1066  if late_payments <= 1.5
  +0.7493  if late_payments > 1.5
tree 4:
  +0.4383  if months_employed <= 12.5 and credit_lines <= 6.5
  -0.0227  if months_employed <= 12.5 and credit_lines > 6.5
  +0.2795  if months_employed > 12.5 and age <= 25.5
  -0.1646  if months_employed > 12.5 and age > 25.5
tree 5:
  -0.1296  if age <= 59.5
  +0.1048  if 59.5 < age <= 73.5
  +0.4337  if age > 73.5
tree 6:
  +0.3009  if (income <= 1.715e+04 or missing)
  -0.0526  if income > 1.715e+04
tree 7:
  +0.0945  if debt

## Round a threshold

Say the business wants the first debt-ratio cut at a round 0.45. `set_cut` moves the cut,
snapping it to the nearest edge of the training bins (so the new threshold is close to,
not exactly, 0.45), and `refit_leaves` updates the values.

In [3]:
import copy

tree0 = itm.get_trees()[0]
node = next(k for k, name in enumerate(tree0["feature_name"]) if name == "debt_ratio")
print("before:", tree0["threshold"][node])

edited = copy.deepcopy(itm)
edited.set_cut(0, node, "debt_ratio", 0.45)
edited.refit_leaves(X_train, y_train)
print("after: ", edited.get_trees()[0]["threshold"][node])
print("edited:", score(edited))

before: 0.4325000047683716
after:  0.4584999978542328
edited: AUC 0.7047, log-loss 0.3062


## Add a rule by hand

A policy says no one over 75 gets credit without review. `add_stump` adds it as a new
one-cut tree; the refit gives it a value learned from the data.

In [4]:
edited.add_stump("age", 75)
edited.refit_leaves(X_train, y_train)
print(edited.rules()[-2:])
print("with the rule:", score(edited))

[(8, ['age <= 75.5'], -0.018828804315996078), (8, ['age > 75.5'], 0.1329799105414049)]
with the rule: AUC 0.7049, log-loss 0.3063


## Monotone constraints

Policy: more months employed must never raise the risk, and more late payments must never
lower it. `monotone_violations` checks the whole model exactly (every combination of
leaves across trees, not a sample of rows); `refit_leaves(..., monotone=...)` enforces
the constraint by re-estimating the leaves, keeping the cuts.

In [5]:
policy = {"months_employed": -1, "late_payments": +1}
for feature, sign in policy.items():
    print(feature, "violations:", len(edited.monotone_violations(feature, increasing=sign > 0)))
edited.refit_leaves(X_train, y_train, monotone=policy)
print("after the constrained refit:")
for feature, sign in policy.items():
    print(feature, "violations:", len(edited.monotone_violations(feature, increasing=sign > 0)))
print(score(edited))

months_employed violations: 1
late_payments violations: 0
after the constrained refit:
months_employed violations: 0
late_payments violations: 0
AUC 0.7059, log-loss 0.3061


## Near-equivalent cuts

`cut_alternatives` lists other cuts that would do almost as well at a node (validation
log-loss within `epsilon`). If a cut has many equally good alternatives, do not read
too much into the particular feature it chose.

In [6]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=1, stratify=y_train)
model = InterleavedTreeClassifier(max_splits=16).fit(X_fit, y_fit)
pd.DataFrame(model.cut_alternatives(X_val, y_val, tree=0, node=0, epsilon=0.01))

,feature,feature_name,threshold,loss,current,delta
0,3,late_payments,0.5000,0.298316,True,0.000000
1,2,debt_ratio,0.5665,0.300525,False,0.007405
2,0,age,27.5000,0.301214,False,0.009715


## The Rashomon view

`RashomonFIGSClassifier` mutates cuts and keeps every structure within `epsilon` of the
best validation loss. `rashomon_importance` gives the range of each feature's importance
across those equally good models. It is research code (`bettertrees.lab`: no API
guarantee).

In [7]:
from bettertrees.lab import RashomonFIGSClassifier

rash = RashomonFIGSClassifier(max_splits=16, n_mutations=60, random_state=0).fit(X_train, y_train)
print(len(rash.rashomon_models()), "models within epsilon of the best validation loss")
imp = pd.DataFrame(rash.rashomon_importance(X_test), index=["min", "mean", "max"]).T
imp.sort_values("mean", ascending=False).round(3)

25 models within epsilon of the best validation loss


,min,mean,max
debt_ratio,0.224,0.270,0.294
late_payments,0.207,0.267,0.368
income,0.131,0.150,0.158
age,0.045,0.122,0.159
credit_lines,0.078,0.094,0.156
months_employed,0.036,0.073,0.149
has_mortgage,0.000,0.025,0.082


Late payments and the debt ratio matter in every equally good model (their minimum stays
high). The mortgage flag can drop to zero: some models that fit just as well do not use
it at all, so its importance in any one model should not be over-read.

(The Rashomon search holds out 20% of the training rows for validation, so its selected
model is fit on less data than the plain ITM above; use it to see the set, not as a
better predictor.)